# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [1]:
import pandas as pd

cars = pd.read_csv('../../Data/mpg.csv')
cars.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [2]:
rows, columns = cars.shape
print("rows:", rows, "| columns:", columns)
print()
cars.info()

rows: 398 | columns: 9

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    object 
 8   name          398 non-null    object 
dtypes: float64(4), int64(3), object(2)
memory usage: 28.1+ KB


**Answer:** only `horsepower` has missing values. It shows 392 non-null out of 398 rows, so **6** are missing.

How many cars from each `origin`? Use `value_counts()`.

In [3]:
print(cars['origin'].value_counts())

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64


USA has 249 cars, Japan 79 and Europe 70. USA is much bigger than the other two, which is why the split uses `stratify=y` to keep the same mix in train and test.

---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [4]:
# 1. fill the blank horsepower values with the median
hp_median = cars['horsepower'].median()
print("median horsepower:", hp_median)
cars['horsepower'] = cars['horsepower'].fillna(hp_median)

# 2. name column is not useful, remove it
cars = cars.drop('name', axis=1)

print("missing values now:", cars.isnull().sum().sum())
print("columns left:", list(cars.columns))

median horsepower: 93.5
missing values now: 0
columns left: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']


*Why `name` cannot help:*

The name is different for almost every car, it works like a label for one row and not like a feature, so there is nothing general the model can learn from it.

---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [5]:
target = 'origin'

X = cars.drop(target, axis=1)
y = cars[target]

print("features:", list(X.columns))
print("X:", X.shape, " y:", y.shape)

features: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X: (398, 7)  y: (398,)


---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [6]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y,
                                          test_size=0.2,
                                          random_state=0,
                                          stratify=y)

print("train rows:", len(X_tr))
print("test rows :", len(X_te))

train rows: 318
test rows : 80


---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [7]:
for col in X_tr.columns:
    low = X_tr[col].min()
    high = X_tr[col].max()
    print(f"{col:<14} min = {low:<8} max = {high:<8} range = {round(high - low, 1)}")

spread = X_tr.max() - X_tr.min()
print("\nlargest range:", spread.idxmax())

mpg            min = 9.0      max = 46.6     range = 37.6
cylinders      min = 3        max = 8        range = 5
displacement   min = 68.0     max = 455.0    range = 387.0
horsepower     min = 46.0     max = 225.0    range = 179.0
weight         min = 1613     max = 5140     range = 3527
acceleration   min = 8.0      max = 24.8     range = 16.8
model_year     min = 70       max = 82       range = 12

largest range: weight


**Answer:** `weight` has the biggest range by far (about 1613 to 5140). Columns like `cylinders` (3 to 8) or `acceleration` (8 to 24.8) are tiny next to it.

Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [8]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)   # learn mean and std from train only
X_te_sc = sc.transform(X_te)       # reuse the same numbers on test

print(X_tr_sc[:3].round(2))

[[ 1.46 -0.85 -1.15 -0.92 -1.57  0.89 -1.37]
 [ 0.56 -0.85 -0.77 -0.42 -0.42  1.48  1.6 ]
 [-1.1   1.49  1.19  1.21  0.94 -1.1  -0.83]]


---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_plain = KNeighborsClassifier()
knn_plain.fit(X_tr, y_tr)

plain_pred = knn_plain.predict(X_te)
plain_acc = accuracy_score(y_te, plain_pred)
print("accuracy without scaling:", plain_acc)

accuracy without scaling: 0.6875


**Now with scaling.** Same model, same `k`, scaled data.

In [10]:
knn_std = KNeighborsClassifier()
knn_std.fit(X_tr_sc, y_tr)

std_pred = knn_std.predict(X_te_sc)
std_acc = accuracy_score(y_te, std_pred)
print("accuracy with scaling:", std_acc)

accuracy with scaling:

 0.75


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

In [11]:
print(f"unscaled : {plain_acc:.4f}")
print(f"scaled   : {std_acc:.4f}")

gap = std_acc - plain_acc
if gap > 0:
    print(f"scaling is better by {gap:.4f} ({gap * 100:.2f} percentage points)")
else:
    print(f"scaling is not better, gap = {gap:.4f}")

unscaled : 0.6875
scaled   : 0.7500
scaling is better by 0.0625 (6.25 percentage points)


*What changed between the two runs:*

Only one thing changed: the columns were put on the same scale. Before scaling, the distance KNN calculates was mostly decided by `weight` because its numbers are in the thousands, so the other columns like `acceleration` or `cylinders` had almost no say. After scaling every column pulls equally, so the neighbours picked are closer in every way, and accuracy went up from 0.6875 to 0.75.

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [12]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
scores = {}

for k in k_values:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr_sc, y_tr)
    scores[k] = accuracy_score(y_te, clf.predict(X_te_sc))
    print(f"k = {k:<3} accuracy = {scores[k]:.4f}")

best = max(scores.values())
best_ks = [k for k in scores if scores[k] == best]
print("\nbest accuracy:", best, "for k =", best_ks)
print("lowest accuracy:", min(scores.values()))

k = 1   accuracy = 0.7375
k = 3   accuracy = 0.7500
k = 5   accuracy = 0.7500
k = 7   accuracy = 0.7500
k = 9   accuracy = 0.7500
k = 11  accuracy = 0.7250
k = 15  accuracy = 0.7375
k = 21  accuracy = 0.7250

best accuracy: 0.75 for k = [3, 5, 7, 9]
lowest accuracy: 0.725


*Best k:* &nbsp;&nbsp; *Its accuracy:*

Does the accuracy change a lot across k, or stay roughly flat?

k = 3, 5, 7 and 9 (same score) &nbsp;&nbsp; 0.75

*Your answers:*

Pretty flat. All values stay between 0.725 and 0.75, so the difference between the best and worst k is only 2.5 points. Picking k did not matter much here compared to scaling the data.